In [ ]:
import json
import pandas as pd
import re

def extract_data (file_path):
    Data = []
    white_space_fix = re.compile(r'\s+')
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:

            doc = json.loads(line)
            categories = doc.get('categories', '')
            if 'cs.' not in categories:
                continue
            update_date = doc.get('update_date', '')
            year = int(update_date[:4])
            cs_cats = [c for c in categories.split() if c.startswith('cs.')]

            if not cs_cats:
                continue
            raw_abstract = doc.get('abstract', '')
            raw_title = doc.get('title', '')
            if not raw_abstract:
                continue

            cleaned_abstract = white_space_fix.sub(' ', raw_abstract.replace('\n', ' ')).strip()
            cleaned_title = white_space_fix.sub(' ', raw_title.replace('\n', '')).strip()

            Data.append({
             'id': doc.get('id'),
             'title': cleaned_title,
             'authors': doc.get('authors', ''),
             'year': year,
             'abstract': cleaned_abstract,
             'cs_cats': cs_cats,
             'first_cat':cs_cats[0],
             'cs_cats_count': len(cs_cats),
            })
    df = pd.DataFrame(Data)
    df = df.dropna(subset=['abstract' , 'cs_cats'])

    return df
df_multi = extract_data('../data/raw/arxiv-metadata-oai-snapshot.json')
print("\n--- Multi-Label Data Structure ---")
print(df_multi[['id', 'first_cat', 'cs_cats', 'cs_cats_count']].head(50))

print("\n--- Distribution of Category Counts ---")
print(df_multi['cs_cats_count'].value_counts().sort_index())


In [ ]:
import os

# Create a 'processed' directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

# Save the DataFrame
file_path = '../data/processed/arxiv_cs_full.parquet'
print(f"Saving 900k+ records to {file_path}...")

# index=False saves space because our default pandas index (0, 1, 2...) is useless
cleaned_df = df_multi.to_parquet(file_path, engine='pyarrow', index=False)

print("Serialization complete. You can now delete the extraction script from your daily workflow.")

In [ ]:
df = pd.read_parquet("../data/processed/arxiv_cs_full.parquet")
print(f"Successfully loaded {len(df)} papers into RAM.\n")

min_year = df["year"].min()
max_year = df["year"].max()

print("min : ", min_year, "| max : ", max_year)

percentage_of_more_cats = (df["cs_cats_count"]>1).mean()*100
print(percentage_of_more_cats, "%")

unique_cat = df["first_cat"].unique()
print(unique_cat)

In [ ]:
import plotly.express as px

print("--- CELL 3: Class Distribution Analysis ---")

# 1. Calculate the frequencies for the top 15
top_15_cats = df['first_cat'].value_counts().head(15).reset_index()
top_15_cats.columns = ['Category', 'Paper_Count']

# 2. Sort ascending so the largest bar pushes to the top of the horizontal chart
top_15_cats = top_15_cats.sort_values(by='Paper_Count', ascending=True)

# 3. Build the Plotly figure
fig = px.bar(
    top_15_cats,
    x='Paper_Count',
    y='Category',
    orientation='h',
    title='Top 15 Primary Computer Science Categories (Class Imbalance Check)',
    labels={'Paper_Count': 'Total Number of Papers', 'Category': 'ArXiv Category'},
    text='Paper_Count',
    color='Paper_Count',          # Adds a heat gradient
    color_continuous_scale='Reds' # Darker red = more papers
)

# 4. Clean up the UI
fig.update_traces(textposition='outside')
fig.update_layout(height=600, width=900, template='plotly_white')

fig.show()

In [ ]:
import plotly.express as px

print("--- CELL 4: Temporal Distribution (Growth of CS) ---")

# Calculate papers per year and sort chronologically
yearly_counts = df['year'].value_counts().sort_index().reset_index()
yearly_counts.columns = ['Year', 'Paper_Count']

# Build the Plotly line chart
fig = px.line(
    yearly_counts,
    x='Year',
    y='Paper_Count',
    title='The Exponential Growth of Computer Science (ArXiv Publications per Year)',
    labels={'Year': 'Publication Year', 'Paper_Count': 'Number of Papers'},
    markers=True
)

fig.update_layout(template='plotly_white', height=500, width=900)
fig.show()